<a href="https://colab.research.google.com/github/kulanigayathree/J26-IT-364/blob/model%2FCoral-reef-health-monitoring/notebooks/C3_Day1b_Cleaner_Composite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Component 3 – Day 1b: Cleaner Sentinel-2 composites (v2)
The Day 1 composites still contained residual clouds, cloud shadows and bright speckles. This notebook improves them:

| Change | v1 (Day 1) | **v2** |
|---|---|---|
| Cloud mask | `cs ≥ 0.6` | **strictest of `cs_cdf ≥ 0.65` / `cs ≥ 0.65` / `cs ≥ 0.6` that keeps ≥ 90% reef coverage** |
| Very cloudy scenes | included | **only `CLOUDY_PIXEL_PERCENTAGE < 50`** |
| Time window | one season | **two clear seasons** (more clear observations) |
| Clear observations | not checked | **pixels with fewer than 3 clear images removed** |
| Labels | label only | **label + purity** (agreement of the four 5 m ACA pixels inside each 10 m pixel) |

**Four exports** (two per site): `S2_<site>_composite_v2.tif`, `ACA_<site>_label_v2.tif`.
Training samples are drawn directly from these rasters in Day 2b. Run the export cell (Step 6) only once.

## Step 1 – Setup and authentication

In [1]:
!pip install -q earthengine-api geemap
import ee, geemap, os
PROJECT_ID = 'coral-c3-2026'
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)
BASE_FOLDER = 'coral_c3'
print('✅ Earth Engine OK:', ee.String('hello reef').getInfo())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 43.4 MB/s eta 0:00:00
✅ Earth Engine OK: hello reef


## Step 2 – Sites and clear-water windows
Same sites as Day 1. Two clear seasons before the 2024 bleaching event are used as the baseline.

In [2]:
sites = {
    'HKD': ee.Geometry.Rectangle([80.085, 6.125, 80.110, 6.150]),   # Hikkaduwa (south-west)
    'PGI': ee.Geometry.Rectangle([81.175, 8.690, 81.235, 8.760]),   # Pigeon Island + Nilaveli (east)
}
windows = {
    'HKD': [('2022-12-01', '2023-04-15'), ('2023-12-01', '2024-03-31')],   # NE monsoon dry season, before 2024 heat stress
    'PGI': [('2022-05-01', '2022-09-30'), ('2023-05-01', '2023-09-30')],   # SW monsoon = calm/clear on east coast
}

## Step 3 – Cleaner composite (v2)

In [9]:
S2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
CS = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
BANDS = ['B2', 'B3', 'B4', 'B8', 'B11']
MIN_OBS = 3
reef_mask = ee.Image('ACA/reef_habitat/v2_0').select('benthic').gt(0)

def s2_collection(aoi, wins, band, thresh):
    date_filter = ee.Filter.Or(*[ee.Filter.date(a, b) for a, b in wins])   # keeps system:index (no merge)
    return (S2.filterBounds(aoi).filter(date_filter)
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 50))
              .linkCollection(CS, [band])
              .map(lambda img: img.updateMask(img.select(band).gte(thresh))))

def s2_composite_v2(aoi, wins, band, thresh):
    col = s2_collection(aoi, wins, band, thresh).select(BANDS)
    n_obs = col.select('B2').count().rename('n_obs')
    img = col.median().divide(10000)
    water = img.normalizedDifference(['B3', 'B8']).gt(0)
    return img.updateMask(water).updateMask(n_obs.gte(MIN_OBS)).clip(aoi), n_obs.clip(aoi)

def reef_coverage(img, aoi):
    r = ee.Image.cat([img.select('B2').mask().And(reef_mask).rename('ok'), reef_mask.rename('reef')]) \
          .reduceRegion(ee.Reducer.sum(), aoi, 10, 'EPSG:32644', maxPixels=1e9).getInfo()
    return (r['ok'] or 0) / max(r['reef'] or 1, 1)

CONFIGS = [('cs_cdf', 0.65), ('cs', 0.65), ('cs', 0.60)]   # strictest first
composites, nobs, chosen = {}, {}, {}
for k in sites:
    n_img = s2_collection(sites[k], windows[k], 'cs', 0.6).size().getInfo()
    print(f'{k}: {n_img} images')
    for band, th in CONFIGS:
        img, n = s2_composite_v2(sites[k], windows[k], band, th)
        cov = reef_coverage(img, sites[k])
        p50 = n.updateMask(reef_mask).reduceRegion(ee.Reducer.median(), sites[k], 10, 'EPSG:32644', maxPixels=1e9).get('n_obs').getInfo()
        print(f'   {band}>={th}:  reef coverage {cov:6.1%}   clear obs (median) {p50}')
        if cov >= 0.90:
            composites[k], nobs[k], chosen[k] = img, n, (band, th)
            break
    if k not in chosen:
        composites[k], nobs[k], chosen[k] = img, n, (band, th)
    print(f'   → {k} uses {chosen[k]}\n')

HKD: 51 images
   cs_cdf>=0.65:  reef coverage  96.3%   clear obs (median) 41
   → HKD uses ('cs_cdf', 0.65)

PGI: 25 images
   cs_cdf>=0.65:  reef coverage  98.1%   clear obs (median) 21.000000000000018
   → PGI uses ('cs_cdf', 0.65)



**Result:** the strictest mask (`cs_cdf ≥ 0.65`) was selected for both sites:
Hikkaduwa 51 images, 96.3% reef coverage, median 41 clear observations;
Pigeon Island 25 images, 98.1% reef coverage, median 21 clear observations.

## Step 4 – ACA labels (10 m, mode) and purity
**Purity** = fraction of the four 5 m ACA pixels in a 10 m pixel that belong to the majority class (0.25–1.0).
1.0 = pure pixel; 0.5 = mixed pixel (e.g. reef edge).

In [4]:
CRS = 'EPSG:32644'
proj10 = ee.Projection(CRS).atScale(10)
CODES = [11, 12, 13, 14, 15, 18]
CLASS_NAMES = {11: 'Sand', 12: 'Rubble', 13: 'Rock', 14: 'Seagrass', 15: 'Coral/Algae', 18: 'Microalgal Mats'}

aca5 = ee.Image('ACA/reef_habitat/v2_0').select('benthic')
label10 = aca5.reduceResolution(ee.Reducer.mode(), maxPixels=16).reproject(proj10).toInt().rename('label')
fracs = [aca5.eq(c).reduceResolution(ee.Reducer.mean(), maxPixels=16).reproject(proj10) for c in CODES]
purity10 = ee.Image.cat(fracs).reduce(ee.Reducer.max()).rename('purity')

for k, g in sites.items():
    h = label10.reduceRegion(ee.Reducer.frequencyHistogram(), g, 10, CRS, maxPixels=1e9).get('label').getInfo() or {}
    p = purity10.updateMask(label10.gt(0)).reduceRegion(ee.Reducer.mean(), g, 10, CRS, maxPixels=1e9).getInfo()
    print(k, {CLASS_NAMES.get(int(float(c)), c): int(v) for c, v in h.items()}, '| mean purity:', round(p.get('purity') or 0, 2))

HKD {'Sand': 527, 'Rubble': 2105, 'Rock': 1699, 'Coral/Algae': 5299} | mean purity: 0.94
PGI {'Sand': 1234, 'Rubble': 890, 'Rock': 3485, 'Seagrass': 23, 'Coral/Algae': 1509} | mean purity: 0.96


**Result:** class counts match Day 1; mean purity 0.94 (Hikkaduwa) and 0.96 (Pigeon Island).

## Step 5 – Visual check (v2 composite)
Compare with the Day 1 composite: clouds, shadows and speckles should be removed.

In [11]:
rgb_vis = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.15}
SITE_TO_SHOW = 'PGI'      # set to 'PGI' and re-run for the other site
centre = sites[SITE_TO_SHOW].centroid(1).coordinates().getInfo()[::-1]
m = geemap.Map(center=centre, zoom=15)
m.add_basemap('SATELLITE')
m.addLayer(composites[SITE_TO_SHOW], rgb_vis, f'S2 composite v2 {SITE_TO_SHOW}')
m.addLayer(nobs[SITE_TO_SHOW], {'min': 0, 'max': 30, 'palette': ['red', 'yellow', 'green']}, 'clear obs count', False)
m

Map(center=[8.725000083863547, 81.20499999999636], controls=(WidgetControl(options=['position', 'transparent_b…

## Step 6 – Export (run once)
Per site: composite (5 bands, int16 × 10000) and label raster (band 1 = label, band 2 = purity × 100).

In [12]:
tasks = []
for k in sites:
    t = ee.batch.Export.image.toDrive(
        image=composites[k].multiply(10000).toInt16().unmask(-9999),
        description=f'S2_{k}_composite_v2', folder=BASE_FOLDER,
        region=sites[k], scale=10, crs=CRS, maxPixels=1e9)
    t.start(); tasks.append(t)
    lab = ee.Image.cat([label10.unmask(0), purity10.multiply(100).round().unmask(0)]).toUint8().clip(sites[k])
    t = ee.batch.Export.image.toDrive(
        image=lab, description=f'ACA_{k}_label_v2', folder=BASE_FOLDER,
        region=sites[k], scale=10, crs=CRS, maxPixels=1e9)
    t.start(); tasks.append(t)
print('Started', len(tasks), 'export tasks')

Started 4 export tasks


**Expected:** `Started 4 export tasks`.

## Step 7 – Export status
Re-run until all four tasks are `COMPLETED` (also visible in the Earth Engine Code Editor **Tasks** tab).

In [13]:
for t in tasks:
    s = t.status()
    print(f"{s['description']:<24} {s['state']:<10} {s.get('error_message', '')}")

S2_HKD_composite_v2      COMPLETED  
ACA_HKD_label_v2         COMPLETED  
S2_PGI_composite_v2      COMPLETED  
ACA_PGI_label_v2         COMPLETED  


All four v2 files are saved in Drive `coral_c3`.
Next: Day 2b (improvements).